# TRACE fork — tiny smoke test

Loads each of the three HMM variants (`hmm1`, `hmm2`, `chmm`), generates one short continuation on a small prompt, and scores with Detoxify.

This is a kernel sanity check — `max_new_tokens=10`, `num_generations=1`, one prompt. Don't cite the numbers; run the driver scripts for anything real (see [README.md](README.md)).

In [ ]:
import os, sys, gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessorList

sys.path.insert(0, os.path.abspath('.'))
from src import utils
from src.logits_processor import HmmGuidedLogitsProcessor
from src.logits_processor_sohmm import SOHmmGuidedLogitsProcessor
from src.logits_processor_chmm import CHMMGuidedLogitsProcessor

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## Tiny knobs

Keep these small — this is a kernel smoke test, not a benchmark.

In [ ]:
PROMPT = 'The weather today is'
MAX_LEN = 10
NUM_GENERATIONS = 1
A = 1.0
WEIGHTS_PATH = 'classifiers/coefficients_nontoxicity.csv'

VARIANTS = [
    ('hmm1', 'models/hmm_gpt2-large_bttf'),
    ('hmm2', 'models/hmm2_gpt2-large_64_bttf'),
    ('chmm', 'models/chmm_gpt-2-large_uniform6_bttf'),
]

## Load the base LM and coefficient weights

In [ ]:
print('Loading GPT-2 Large + tokenizer...')
gen_model = AutoModelForCausalLM.from_pretrained('gpt2-large').to(device).eval()
gen_tokenizer = AutoTokenizer.from_pretrained('gpt2-large', padding_side='left')
gen_tokenizer.pad_token = gen_tokenizer.pad_token or gen_tokenizer.eos_token

weights_tensor = utils.load_weights(WEIGHTS_PATH, device=device)
print(f'weights shape: {tuple(weights_tensor.shape)}')

## Helper: load a variant, generate one continuation, free memory

Signatures across the three processors are uniform: `(hmm_model, expectation_cache, a, tokenizer, ...)` — see [agent.md](agent.md) for the dispatch table.

In [ ]:
LOADERS = {
    'hmm1': (utils.load_hmm_model,  HmmGuidedLogitsProcessor),
    'hmm2': (utils.load_sohmm_model, SOHmmGuidedLogitsProcessor),
    'chmm': (utils.load_chmm_model,  CHMMGuidedLogitsProcessor),
}

def run_variant(variant, model_path):
    load_fn, proc_cls = LOADERS[variant]
    model = load_fn(model_path, device=device)
    model.set_weights(weights_tensor)
    expectation_cache = model.compute_backward_expectation(T=MAX_LEN)
    processor = proc_cls(
        hmm_model=model,
        expectation_cache=expectation_cache,
        a=A,
        tokenizer=gen_tokenizer,
    )

    inputs = gen_tokenizer([PROMPT], return_tensors='pt', padding=True).to(device)
    processor.configure_for_prompts(inputs.input_ids)
    with torch.no_grad():
        out = gen_model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            logits_processor=LogitsProcessorList([processor]),
            max_new_tokens=MAX_LEN,
            num_return_sequences=NUM_GENERATIONS,
            do_sample=True, top_p=0.9, temperature=1.0,
            pad_token_id=gen_tokenizer.pad_token_id,
        )
    continuation = gen_tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )

    del model, processor, expectation_cache
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return continuation

## Generate baseline + each of the three variants

In [ ]:
results = {}

# Baseline: unguided GPT-2
inputs = gen_tokenizer([PROMPT], return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    out = gen_model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=MAX_LEN,
        num_return_sequences=NUM_GENERATIONS,
        do_sample=True, top_p=0.9, temperature=1.0,
        pad_token_id=gen_tokenizer.pad_token_id,
    )
baseline_cont = gen_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
results['baseline'] = baseline_cont
print('baseline:', repr(PROMPT), '+', repr(baseline_cont))

for name, path in VARIANTS:
    print()
    print('>>>', name, '(', path, ')')
    results[name] = run_variant(name, path)
    print(name + ':', repr(PROMPT), '+', repr(results[name]))

## Score with Detoxify

Same toxicity scorer `src/score.py` uses (`Detoxify('original')`).

In [ ]:
from detoxify import Detoxify

detox = Detoxify('original')
rows = []
for variant, continuation in results.items():
    toxicity = float(detox.predict(continuation)['toxicity'])
    rows.append({
        'variant': variant,
        'continuation': continuation,
        'toxicity': toxicity,
    })

df = pd.DataFrame(rows).sort_values('toxicity').reset_index(drop=True)
df

If any row fails to load (missing checkpoint, for example), drop it from `VARIANTS` above and re-run. For the full multi-prompt pipeline, use `scripts/gen_all.sh` + `scripts/score_all.sh` + `scripts/analyze.sh` — see [README.md](README.md).